# Model Training - LSTM

Entraînement du modèle LSTM pour prédictions de prix

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data.collector import DataCollectorFactory
from src.data.preprocessor import DataPreprocessor
from src.models.lstm_model import LSTMModel

In [ ]:
# Collecter et prétraiter
collector = DataCollectorFactory.create_collector('EURUSD')
df = await collector.fetch_historical_data('EURUSD', 'daily', 1000)

preprocessor = DataPreprocessor()
df = preprocessor.clean_data(df)
df = preprocessor.add_technical_features(df)
df_normalized, scaler_params = preprocessor.normalize_data(df)

print(f'Data shape: {df.shape}')

In [ ]:
# Créer sequences d'entraînement
X, y = preprocessor.create_training_data(df_normalized, lookback=60, horizon=5)
print(f'X shape: {X.shape}, y shape: {y.shape}')

# Split train/val
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
# Créer et entraîner le modèle
model = LSTMModel(
    input_shape=(60, X.shape[2]),
    lstm_units=128,
    dropout_rate=0.2
)

history = model.train(
    X_train, y_train,
    X_val, y_val,
    epochs=50,
    batch_size=32
)

print('Training completed!')

In [ ]:
# Sauvegarder le modèle
model.save('models/lstm_eurusd_60_5.h5')
print('Model saved!')

In [ ]:
# Faire des prédictions
predictions = model.predict(X_val[:10])
print(f'Predictions: {predictions[:5].flatten()}')

In [ ]:
import matplotlib.pyplot as plt

# Visualiser les prédictions
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('LSTM Training History')
plt.show()